# How hard will the ground shake at distance R from a magnitude M earthquake?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2FT1_how_hard_will_it_shake.ipynb).

Somebody designing a hospital has to answer this with a number before the concrete is poured. Every
seismic building code in the world rests on a **ground-motion prediction equation**: a formula that
takes a magnitude, a distance and a description of the ground underfoot, and returns the peak
acceleration to build for. It is fitted to recordings of real earthquakes, and the fit is never
close — the same magnitude at the same distance can shake one site several times harder than
another.
The width of that miss, not the middle of it, is what a building code actually spends money on.

There are now enough recordings to fit something far more flexible than a formula, and papers
reporting that machine learning beats the classical equation appear every year. This project is
about whether that is true. The answer turns out to depend almost entirely on one decision that has
nothing to do with the model: **which records you hide from yourself before you score it.**

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## How this notebook is different

This is a **project track**. It is not a weekly notebook and it does not behave like one.

A weekly notebook shows you a move, walks you through it, and then asks you to make it once
yourself. This one loads the data and reproduces the one result the field already agrees on — the
classic ground-motion equation, its coefficients and its scatter — and then stops helping. From
there on every section is a sentence describing what to find out and an empty cell to find it out
in. There is no worked example above to pattern-match against, because on a real question there
never is one.

**There is exactly one self-check in this notebook, and it is on the data loading.** After that,
nothing tells you whether you are right. That is not an oversight and it is not laziness: past the
loading step there is no single right answer here, so a cell that said `assert` would be lying to
you about how research works. What replaces it is the thing researchers actually use — a result you
can get two ways, a number you can predict before you compute it, and a claim you can try to break.

**And it does not close.** The last section is a question this course does not know the answer to.
Everything above it is scaffolding; that question is the project.

## What you'll be able to do

**The science.** Fit the equation that seismic building codes are built on, say how wrong it
typically is and in what units that matters, and then decide — from your own measurements, not from
a paper's claim — whether a machine-learning model genuinely predicts shaking better or only appears
to under a careless test.

**The skills.** Turn raw columns into the features a physical model needs. Split a dataset four
different ways and see the score change without the model changing at all. Put an interval on the
*difference* between two models, so that "better" is a claim with a number attached.

**The four questions, in order:**

1. How much of the shaking can one straight line explain?
2. How do you split train from test, and does the choice change the answer?
3. Does a flexible model beat the physics, or only look like it?
4. What is the leftover scatter made of?

The open question at the end is not on that list. It is the project; the four above are what you
build to reach it.

## Setup

The Engineering Strong Motion database publishes a *flatfile*: one row per recording of one
earthquake at one instrument, with the earthquake, the instrument, the ground beneath it and the
processed shaking all on the same line. There is no key and no login, and the whole thing comes
back from one URL.

**Read this before you go on.** Four things about the file decide what you can honestly do with it,
and all four are measurable rather than assumed:

- The file is **semicolon-delimited**, so `pd.read_csv` needs `sep=";"`. Read it the usual way and
  you get a single column with the whole row inside it.
- **`rotd50_pga` is in cm/s²**, and everything in earthquake engineering is quoted in *g*. Dividing
  by 981 is not a formatting choice; forget it and every number you report is out by a
  factor of a thousand.
- **Distance is not one column.** `jb_dist` is the distance to the rupture surface, which is what a
  ground-motion equation really wants, and it is missing on 84% of the
  rows because it requires a fault model somebody had to build. `epi_dist`, the distance to the
  epicentre, is always there. Filling one from the other is a compromise, and it is yours to defend.
- **`preferred_vs30_m_s`** is how fast a shear wave travels through the top thirty metres of ground
  — the standard one-number description of a site, low for soft mud and high for hard rock. It is
  present on 74% of the usable rows and blank on the rest.

The query is pinned to a magnitude floor of 4.5 and the window 1970-01-01 to
2026-01-01, and it takes the better part of a minute. The database grows as new earthquakes are
processed, so your counts may differ from the ones printed in this notebook by a few.

In [ ]:
import numpy as np
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from torch import nn
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cache_name):
    """Read the live flatfile; fall back to the copy stored with the course."""
    # Ask the live archive first. If it is down, or you are offline, read the copy stored with
    # the course instead, so the notebook still runs.
    try:
        return pd.read_csv(url, sep=";")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cache_name, sep=";")

FIELDS = ("esm_event_id,event_time,ev_latitude,ev_longitude,ev_depth_km,ev_nation_code,"
          "fm_type_code,ml,mw,network_code,station_code,st_latitude,st_longitude,"
          "st_nation_code,preferred_vs30_m_s,preferred_ec8_code,epi_dist,jb_dist,"
          "quality_class,late_triggered_event_01,rotd50_pga,rotd50_pgv")
ESM = (f"https://esm-db.eu/esmws/flatfile/1/query?min-magnitude=4.5"
       f"&starttime=1970-01-01&endtime=2026-01-01&include-fields={FIELDS}")

records = load(ESM, "trackT1_esm_flatfile.csv.gz")
print("the flatfile as it arrives:", records.shape)
print(records[["event_time", "mw", "epi_dist", "preferred_vs30_m_s", "rotd50_pga"]].head())

Five columns are built from the raw ones, and then two filters are applied. The first throws away
rows the database itself flags as unreliable. The second cuts to the records a ground-motion
equation is honest about: magnitude 4.0 and up, within 200 km, and with the site
actually measured.

In [ ]:
records["mag"] = records["mw"].fillna(records["ml"])
records["R"] = records["jb_dist"].fillna(records["epi_dist"])
records["pga_g"] = records["rotd50_pga"] / 981.0
records["vs30"] = records["preferred_vs30_m_s"]
records["station"] = records["network_code"] + "." + records["station_code"]

usable = records[(records["pga_g"] > 0)
                 & records["mag"].notna()
                 & (records["quality_class"] != "BAD")
                 & (records["late_triggered_event_01"].fillna(0) == 0)]

shaking = usable[(usable["mag"] >= 4.0)
                 & (usable["R"] <= 200)
                 & usable["vs30"].notna()].reset_index(drop=True).copy()

shaking["log_pga"] = np.log10(shaking["pga_g"])
shaking["log_r"] = np.log10(np.sqrt(shaking["R"] ** 2 + 8 ** 2))
shaking["log_vs30"] = np.log10(shaking["vs30"])

print("rows from the service: ", len(records))
print("passing the quality cut:", len(usable))
print("usable for a model:    ", len(shaking))

In [ ]:
assert "rotd50_pga" in records.columns and "preferred_vs30_m_s" in records.columns, \
    "a column this project needs is missing — the query or the service's schema changed"
assert 14000 < len(shaking) < 20000, \
    "expected about 16930 usable records; a very different number means a filter missed"
assert shaking["pga_g"].max() < 3, \
    "a peak acceleration above 3 g means the cm/s^2 to g conversion did not happen"
print(f"✓ the data — {len(records)} rows from the service, {len(shaking)} usable for a "
      f"ground-motion model, from {shaking['esm_event_id'].nunique()} earthquakes recorded at "
      f"{shaking['station'].nunique()} stations")

### And that is the last self-check in this notebook

The pipeline is now trustworthy: the file is the file, the filters are the filters, the units are
in g. Everything from here is yours, and nothing will tell you when you have it right.

## How much of the shaking can one straight line explain?

Before any model, look at what you have. Two given figures: where the recordings are, and how the
shaking falls off with distance.

In [ ]:
coast = pd.read_csv(CACHE + "/coastlines.csv")

plt.plot(coast.lon, coast.lat, color="0.6", lw=0.6)
plt.scatter(shaking["st_longitude"], shaking["st_latitude"], s=2, color="0.35",
            label="recording stations")
plt.scatter(shaking["ev_longitude"], shaking["ev_latitude"], s=6, color="firebrick",
            label="earthquakes")
plt.xlim(-12, 48)
plt.ylim(30, 50)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
inside = (shaking["st_longitude"].between(-12, 48)
          & shaking["st_latitude"].between(30, 50))
plt.title(f"{len(shaking)} recordings; {inside.sum()} of them inside this box")
plt.legend(loc="lower left", fontsize=7)
plt.show()

print(shaking["st_nation_code"].value_counts().head(6).to_dict())

Now the physics. Shaking grows with magnitude and dies away with distance, and both effects span
factors of thousands, so neither is visible on ordinary axes.

**Log axes:** When the values span factors of a thousand, plot the exponents instead and a curve becomes a line.

In [ ]:
bands = [(4.0, 5.0), (5.0, 6.0), (6.0, 9.0)]
shades = ["0.75", "0.45", "firebrick"]

for band, shade in zip(bands, shades):
    low, high = band
    rows = shaking[(shaking["mag"] >= low) & (shaking["mag"] < high)]
    plt.scatter(rows["R"], rows["pga_g"], s=2, color=shade,
                label=f"M {low}-{high}  (n = {len(rows)})")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("distance to the earthquake (km)")
plt.ylabel("peak ground acceleration (g)")
plt.title(f"How shaking falls off with distance (n = {len(shaking)})")
plt.legend(fontsize=7)
plt.show()

Three clouds, one above the other, each falling away in a straight line. That is what a ground-motion
equation is: a straight line in the right coordinates. Notice what a log axis cannot draw —
63 of these records sit at exactly R = 0 km and are silently missing
from the figure, which is the same problem the formula below has to solve.

```
log10 PGA  =  b0  +  bM * M  +  bR * log10(sqrt(R^2 + h^2))  +  bV * log10(Vs30)
```

Three of those terms are just columns you already have, put on log axes. The fourth is the one piece
of engineering in the formula: `sqrt(R^2 + h^2)` instead of `R`. An earthquake is not a point, so at
zero distance `log10(R)` would be minus infinity and the predicted shaking infinite; `h` is a
made-up depth of a few kilometres that stops that happening. Sweeping it from 4 to 12 km moves the
fit by 0.0113 in R² — smaller than the spread the four splits below will produce
without touching the model at all — so 8 km is used here, and you may refit it.

**Linear regression:** Draw the best straight line. Best means the smallest total miss.

In [ ]:
FEATURES = ["mag", "log_r", "log_vs30"]
X = shaking[FEATURES]
y = shaking["log_pga"]

gmpe = LinearRegression().fit(X, y)
residual = y - gmpe.predict(X)

print("log10 PGA[g] = %.3f %+.3f*M %+.3f*log10(sqrt(R^2 + %d^2)) %+.3f*log10(Vs30)"
      % (gmpe.intercept_, gmpe.coef_[0], gmpe.coef_[1], 8, gmpe.coef_[2]))
print("R2 on the records it was fitted to:", round(gmpe.score(X, y), 3))
print("sigma, the typical miss, in log10 units:", round(residual.std(), 4))

Those four coefficients are the result this notebook hands you, and they are the shape a
seismologist expects: shaking rises with magnitude, falls with distance faster than one over R, and
falls as the ground gets stiffer. Everything after this point is yours.

**Baseline:** Write the dumbest rule you can, first. Any model that cannot beat it is decoration.

### ✏️ Your turn 1

Put a number on what the physics bought, in units an engineer would recognise.

The dumbest possible model ignores magnitude, distance and site and always guesses the average
log shaking. Its typical miss is just the spread of `y` itself. The fitted model's typical miss is
the spread of `residual`. Both are in log10 units, and a miss of 1.0 in log10 means "wrong by a
factor of 10", so `10 ** miss` turns each into a factor.

Print both misses and both factors, and the ratio between the two factors.

Then, in one printed sentence: every one of the 16,930 records was used to choose those
four coefficients, so what has that R² *not* told you, and what would you have to do to find out?

In [ ]:
# ← your answer here



## How do you split train from test, and does the choice change the answer?

**Train/test split:** Hide some data from yourself, then check.

That sentence is one line of code and it hides the entire difficulty of this project. Hide *which*
data? The obvious answer — a quarter of the rows, picked at random — is the one every introductory
tutorial gives, and on this dataset the rows are not independent things. The median earthquake here
contributes 4 rows and the largest contributes 214; the
median station contributes 2 and the busiest 252.
What that does to a split made at random is the first thing to measure.

There are at least four defensible answers, and they are the one real decision in this track:

- **at random** — every record is its own independent thing;
- **by event** — hide whole earthquakes, so the model has never seen this rupture;
- **by station** — hide whole recording sites, so the model has never seen this patch of ground;
- **across a border** — train everywhere except Italy and test in Italy, which is what you are
  really doing whenever you apply a European equation in California.

This is the idea the model-selection week called *leakage*: any route by which information about the
data you are scoring on reaches the model before you score it. Below are the helpers you will need;
the four splits are yours to build.

Two scores, not one, and the reason matters. **R² is a ratio**: the miss divided by the spread of
the very test set it was measured on. Change the test set and you change the denominator, so four
R² from four different splits are four different quantities and cannot be laid side by side.
**RMSE is the miss itself**, in log10 units of PGA, and those units mean the same thing in every
test set. Report both, and where they disagree, believe the one whose units you can name.

In [ ]:
def hold_out_quarter(labels, seed=88):
    """True for the rows to TRAIN on, False for the rows held out.

    Whole labels go into the test set, in random order, until a quarter of the records are gone —
    so `labels` is where you say what one independent thing is.
    """
    # 1. Count how many records each label owns, then shuffle that list. Without the shuffle the
    #    biggest earthquakes would be held out every time, which asks a different question.
    counts = pd.Series(labels).value_counts().sample(frac=1, random_state=seed)
    held_out = []
    rows_out = 0
    # 2. Move whole labels into the test set until a quarter of the RECORDS have gone. Counting
    #    records rather than labels keeps the test set the same size whichever column you split on.
    for name, n_rows in counts.items():
        if rows_out >= 0.25 * len(labels):
            break
        held_out.append(name)
        rows_out = rows_out + n_rows
    # 3. Every record carrying a held-out label leaves together. That is the point of splitting
    #    this way: if two records from one earthquake sat on opposite sides of the split, the
    #    model could half-remember the answer instead of having to predict it.
    return ~np.isin(labels, held_out)


def r_squared(predicted, actual):
    """The fraction of the up-and-down variation a prediction accounts for."""
    return 1 - ((actual - predicted) ** 2).sum() / ((actual - actual.mean()) ** 2).sum()


def rmse(predicted, actual):
    """The typical miss, in the units of the thing being predicted."""
    return np.sqrt(((actual - predicted) ** 2).mean())


def held_out_predictions(model, is_train):
    """Fit the model on the training records; hand back what it predicts for the held-out ones.

    `model` is either a scikit-learn estimator — anything with `.fit` and `.predict` — or a plain
    function of `is_train` that does its own fitting and returns the held-out predictions. The
    second form is the one a torch model needs, and accepting both is what lets every model in
    this notebook be scored by the same call instead of two.
    """
    # A scikit-learn model is fitted and then asked; a torch model brings its own training loop
    # and so arrives as a function. Asking whether the object has a `.fit` tells the two apart.
    if hasattr(model, "fit"):
        # The model is shown the training records only. It never sees `X[~is_train]` while it is
        # learning, which is what makes the score below a test rather than a memory check.
        model.fit(X[is_train], y[is_train])
        return model.predict(X[~is_train])
    return model(is_train)


def held_out_r2(model, is_train):
    """Fit on the training records, score on the held-out ones."""
    return r_squared(held_out_predictions(model, is_train), y[~is_train])


def held_out_rmse(model, is_train):
    """Fit on the training records, and how far off it typically is on the held-out ones."""
    return rmse(held_out_predictions(model, is_train), y[~is_train])

### ✏️ Your turn 2

Build the four splits, as a dictionary of name → mask, so that every later section can loop over
the same four. Three of them come from `hold_out_quarter` with a different `labels` argument; the
fourth is a comparison you write yourself.

    "at random"       hold_out_quarter(np.arange(len(shaking)))
    "by event"        hold_out_quarter(shaking["esm_event_id"])
    "by station"      hold_out_quarter(shaking["station"])
    "across a border" (shaking["st_nation_code"] != "IT").values

For each split print six things: how many records are in the training and the test set; what
**fraction of the held-out records share their earthquake with a record in the training set**;
what fraction **share their station**; the held-out R²; and the held-out RMSE. Print the standard
deviation of `y` in each test set too — that is the denominator each R² was divided by, and it is
the reason the two scores will not rank the four splits the same way.

Draw the four R² values and the four RMSE values as two bar charts side by side.

Then print two sentences: which one of these four splits would you put in a paper as "the
equation's predictive accuracy", and what are the other three measuring instead?

In [ ]:
# ← your answer here



### ✏️ Your turn 3

Three paragraphs, quoting **your own numbers** — the four R², the four RMSE, the four test-set
standard deviations and the two 'share' fractions.

1. Only one of the four splits gives a number you could honestly call this equation's predictive
   accuracy. Say which, and say what each of the other three has let in, using the share fractions
   as your evidence rather than as an assertion.
2. Now check what those splits actually did to the score. Rank the four splits by R², then rank
   them by RMSE, and account for the difference between the two rankings using the test-set
   standard deviations — one of these scores is a ratio and one is not. Say plainly whether the
   split you called leaky came out highest, and if it did not, explain why not rather than
   explaining it away.
3. A split that lets information through does not automatically inflate a score; it inflates the
   score of a model that can use the information. Say what that implies about this particular
   equation, and predict what should happen to the spread of these four numbers when you give a
   model more freedom in the next section.

*(Double-click this cell and replace this line with your answer.)*

## Does a flexible model beat the physics, or only look like it?

Now give a model the same three columns and much more freedom.

**Random forest:** Ask a hundred slightly different trees and take a vote. The rock week used it to choose between
labels; here the same forest predicts a number instead, which in scikit-learn means
`RandomForestRegressor` in place of `RandomForestClassifier` and changes nothing else about how it
is called.

**Neural network:** A stack of the logistic regressions you already know.

Neither knows any seismology. Neither has been told that shaking falls off with distance. Both have
enough freedom to notice things about the 16,930 particular records they are shown.

### Predict before you run

You are about to score a random forest against the straight line, on the same three columns, under
each of your four splits. Commit to two numbers first: how much R² the forest gains over the
equation when the held-out quarter is random, and how much it gains when the held-out quarter is
whole stations. A wrong guess you committed to is worth more than a right answer you were shown.

In [ ]:
my_random_gain = 0.10
my_station_gain = 0.10

print("I think the forest gains", my_random_gain, "R2 under a random split and",
      my_station_gain, "under a station split")

### ✏️ Your turn 4

Score `RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=0)` against
`LinearRegression()` on every one of your four splits, using `held_out_r2` for both so that the
only thing changing between rows is the split.

Print, per split: the equation's R², the forest's R², the difference, and both RMSE values so the
row is also readable in the units of the thing predicted. Draw the four differences as a bar chart
with zero marked, so the sign is visible.

Then print `forest.feature_importances_` beside `FEATURES` for a forest fitted on everything.

Finish with three printed sentences on your own numbers. Say what the forest is doing that the
straight line cannot. Then check the obvious explanation against your own share fractions: if the
forest's advantage came from records that share an earthquake or a station with the training set,
what should the advantage be on the split where the share-a-station fraction is zero, and what is
it? Last, say which of your four numbers you would have quoted if you had only ever run the first
one.

In [ ]:
# ← your answer here



### ✏️ Your turn 5

Now the model the open question is actually about: a small neural network on the same three columns.

**The contract, which is all you are given.** Write a function `network_predictions(is_train)` that
trains a network on the training records and hands back its predictions for the held-out ones as a
plain numpy array — so that `held_out_r2(network_predictions, is_train)` scores it by exactly the
call that scored the equation and the forest. The name and that one argument are what the rest of
the notebook depends on; everything inside is yours.

The design, in words. The calls are the waveform week's, and every one of them is in the summary
table at the foot of this notebook.

- Standardise the three features first. Fit the scaler on the training rows **only** and apply it
  to the held-out rows — fitting it on everything is leakage, of a small and famous kind.
- Two hidden layers of 32 units with a rectifier between them, and one output. Torch wants
  float32 tensors, and the targets as a column rather than a row.
- Mean-squared-error loss, Adam at a learning rate of 0.01, 200 passes over the training data
  in reshuffled minibatches of 512.
- Seed the network before you build it, so that rerunning the cell gives you the same answer twice.
- Bring the predictions back out of torch as a flat numpy array.

Score it on all four splits beside the equation and the forest, and print the three R² values per
split. It trains in a second or two per split on a laptop; if the third decimal moves when you rerun
it, that is the random start, not a mistake.

Then print two or three sentences answering the question this track exists for, on your own
numbers: does the network beat the equation? Say under which splits it does and which it does not,
and say what your answer would have been if the only split you had run were the random one.

In [ ]:
# ← your answer here



### Is the difference bigger than the noise in the test set it was measured on?

One thing is still missing before any of this is a claim. A difference in R² is itself a
measurement, made on one particular held-out set, and it has a spread like anything else. The
station-split gap you just printed is a small number; quoting it with no width beside it is
reporting a coin flip as a tendency.

**Bootstrap:** Ask the data the same question a thousand times, using a different random slice of itself each time.

**Confidence interval:** Not one number but the range your number would have wandered over, had the world rolled differently.

### ✏️ Your turn 6

Put an interval on the forest's advantage, for the **station** split, by resampling the held-out set
itself.

**The contract.** Produce `gaps`, an array of 2000 values, each one (forest R² − equation R²)
recomputed on a resampled version of the same held-out set — with both models fitted once, on the
real training set, and then left alone. You are resampling what you *scored on*, not what you
trained on.

Two things decide whether it is right, and both are yours to get right:

- **Resample stations, not rows.** The held-out records are no more independent than the rest of
  the file; two records from one station carry nearly one station's worth of information. So one
  draw is a whole station's block of positions, drawn with replacement until you have as many
  blocks as there are held-out stations. Resampling rows would give you an interval several times
  too narrow, which is the whole reason this section exists.
- **Refit nothing.** The predictions are computed once, before the loop. A bootstrap that refits
  inside the loop is measuring something else, and takes an hour.

Report the 2.5th and 97.5th percentiles of `gaps`, and the fraction of the 2000 resamples in
which the forest's advantage is zero or negative. Draw the 2000 differences as a histogram with
zero and your observed difference marked. Then do the whole thing again for the random split.

Finish with two printed sentences on your own two intervals. Is the advantage you measured in *Your
turn 4* bigger than the noise in the test set it was measured on, and does your answer depend on
which split you ask it about? Be careful what you claim: with the fits frozen, this interval covers
the luck of **which held-out stations you happened to score on**. It says nothing about which
stations went into the training set — changing that would refit both models, and is a different
experiment.

In [ ]:
# ← your answer here



## What is the leftover scatter made of?

A ground-motion equation's sigma is not a nuisance — it is the number seismic hazard analysis
actually consumes, because a building code asks for the shaking that is exceeded once in five
hundred years, and that lives in the tail. Nothing you have fitted today moved it. Before asking
what could, it is worth knowing what it is made of, because sigma splits into two pieces with
completely different meanings: how much whole *earthquakes* come out above or below the equation,
and how much *individual recordings* scatter within one earthquake.

### ✏️ Your turn 7

Split the equation's own scatter into the part that belongs to whole earthquakes and the part that
belongs to individual recordings.

1. `shaking["residual"] = y - gmpe.predict(X)`.
2. The average residual of each earthquake is `shaking.groupby("esm_event_id")["residual"].mean()`.
   Put it back on every row with
   `shaking["event_term"] = shaking["esm_event_id"].map(event_mean)` — `map` looks each row's event
   id up in that table of averages.
3. `shaking["within_event"]` is the residual minus the event term.
4. Print the standard deviation of all three: sigma, tau (the event terms) and phi (what is left).
   Print `tau ** 2 + phi ** 2` next to `sigma ** 2` as well.

Draw the event terms and the within-event residuals as two histograms on the same axes.

Then print two or three sentences: which of the two pieces is bigger on your numbers, what a model
would have to know in order to shrink each one, and which of the three models you have fitted today
had any chance of shrinking either.

In [ ]:
# ← your answer here



## The question, answered

For a magnitude 6.0 earthquake 20 km away on ground with Vs30 = 500 m/s, this
fit says **0.067 g** — and its own scatter says the true value will lie between
0.009 g and 0.499 g nineteen times in twenty, which is 1.96 sigma and
not the 2 sigma it is tempting to round it to. That is a factor of 7.5 either
side of the middle, and about 56 from one end of the range to the other — from
four coefficients fitted to 16,930 recordings of 2,182 earthquakes.

The width is the answer. A building code cannot use the middle of that range; it uses the tail, so
the quantity worth improving is sigma, and sigma is what nothing in this notebook moved.

## What track T1 leans on

**The question.** How hard will the ground shake at distance R from a magnitude M earthquake?

Nothing here is new. These are the weeks to look back at while you work, and the wording is the course's own. It is a long table because this track reaches across six of them — from a straight line to a neural network — which is also why it is the one that will take you longest.

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Linear regression** | Draw the best straight line. Best means the smallest total miss. |
| **Log axes** | When the values span factors of a thousand, plot the exponents instead and a curve becomes a line. |
| **Baseline** | Write the dumbest rule you can, first. Any model that cannot beat it is decoration. |
| **Train/test split** | Hide some data from yourself, then check. |
| **Leakage** | Any route by which information about the data you are scoring on reaches the model before you score it. |
| **Random forest** | Ask a hundred slightly different trees and take a vote. |
| **Neural network** | A stack of the logistic regressions you already know. |
| **Bootstrap** | Ask the data the same question a thousand times, using a different random slice of itself each time. |
| **Confidence interval** | Not one number but the range your number would have wandered over, had the world rolled differently. |

### Code you will reach back for

| Function | What it does |
|---|---|
| `np.log10(values)` | the exponent of every value at once — what turns a power law into a straight line |
| `np.sqrt(x) / np.mean(x)` | the square root and the average — the two halves of a typical miss |
| `table.groupby(column)` | split the table into one group per value |
| `column.value_counts()` | how often each value appears |
| `LinearRegression().fit(x, y)` | find the straight line with the smallest total miss |
| `model.coef_[0]` | the slope of the fitted line |
| `model.intercept_` | where the fitted line crosses zero |
| `model.predict(x)` | what the fitted line says y should be at each x |
| `model.score(x, y)` | R2 — the fraction of the up-and-down variation the line accounts for |
| `table.sample(frac=1, random_state=n)` | shuffle a table; random_state fixes the shuffle so everyone gets the same one |
| `StandardScaler()` | put every column on the same footing, so a distance across them means something |
| `filler.fit_transform(X_train) / filler.transform(X_test)` | learn the fill values on the training set only, then apply the same ones to the test set |
| `forest.feature_importances_` | one number per column: how much of the forest's decisions it carried |
| `plt.bar(x, heights) / plt.barh(labels, values)` | one bar per item; barh when the labels are words |
| `torch.tensor(array)` | hand an array to PyTorch so it can be learned from |
| `torch.manual_seed(n)` | fix the random start, so a training run repeats |
| `nn.Sequential(layers)` | a model that runs the layers you hand it, in order |
| `nn.Linear(in, out)` | one layer of weighted sums — a row of perceptrons |
| `nn.ReLU()` | the activation — the bend that makes a stack more than one straight line |
| `nn.MSELoss()` | the loss: the average squared miss, the same one that fits a straight line |
| `torch.optim.Adam(model.parameters(), lr=)` | the thing that rolls the weights downhill |
| `loss.backward() / optimiser.step() / optimiser.zero_grad()` | work out which way each weight should move, move them, then clear the slate |
| `torch.randperm(n)` | a shuffled order, for going through the data differently each epoch |
| `tensor.detach().numpy()` | take the numbers back out of PyTorch |
| `np.percentile(values, [2.5, 97.5])` | the two values that cut off the bottom and top 2.5% — a 95% interval |

## What your project must contain

Five sections, empty below, required of **every** EPS 88 project regardless of track. They are
headed here so the shape of a good answer is visible while you work. Fill them in as you go; they
are not a write-up you do at the end.

### ✏️ 1 · A one-sentence answer

Your claim and its uncertainty, in one sentence, at the top of your report. If you cannot put a
number and a range in it, you do not have a result yet.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 2 · The trivial baseline

Before any statistic, state the dumbest answer to your question and what it gives. Every later
number is reported against it.

On this track the trivial baseline is "always guess the average", and its held-out R² is zero by
construction — which is exactly what makes it useful, because every R² you report afterwards is a
statement about how far past it you got. Quote it, and quote it again in whatever units your
question is really in.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 3 · Split by structure

Earth data are correlated in space and in time, so a random split puts tomorrow in the training set
and yesterday in the test set, and the score is a lie. Split by time, or by region, and say which
you chose and why.

This track is *about* that choice, so this section carries more weight here than anywhere else in
the course. Name the unit you treated as independent, show the number that told you the other units
were not independent, and report what the score did when you changed it. A project on this track
that reports one split has not done the project.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 4 · What I got wrong

What failed, and what you believed before it failed. Honest failure is graded; a faked success is
not. Your *Predict before you run* guess belongs here if it was wrong.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 5 · AI disclosure

Which tool, what you asked it, what you changed in what it gave you, and how you checked that the
result was true.

*(Double-click this cell and replace this line with your answer.)*

## The open question

> **Does a neural network actually beat a well-designed parametric GMPE, or does it only look that way under a random split?**

Nobody grading this knows the answer, and neither does the literature. Everything above is the
scaffolding; this is the project.

Here is what is established by the notebook you have just run, and it is less than it looks — stated
as quantities, because the values are the ones you measured and they are yours to read off your own
output rather than mine. Under a random split both flexible models gain R² over the equation, and
the random split's own bootstrap interval excludes zero, so that gain is real; it is just not a gain
at anything useful, because almost every record being scored shares an earthquake and a station with
the training set. Take the shared earthquakes away and the gain shrinks. Take the shared stations
away and it shrinks again, to a number whose interval straddles zero. Send the model across a border
and the sign flips: the four-coefficient equation wins outright, in a country neither model has
seen. Four splits, one model, and the conclusion changes with the split.

What is **not** established is why, or whether it has to be that way. Four directions, none of them
worked out here:

1. **Give the flexible model something the equation does not have.** Every model in this notebook
   saw the same three columns, so the most a network could do was rearrange information the
   equation already used. The flatfile carries the style of faulting, the depth, the event and
   station coordinates and thirty-six spectral periods. Add a feature that is genuinely new physics,
   and re-run all four splits. Does the advantage survive the border this time?
2. **Fit the ergodic assumption instead of ignoring it.** Modern ground-motion models add an
   explicit term per site and per region — a *non-ergodic* model. Your event terms from *Your turn
   7* are a first draft of exactly that. What happens to tau and phi if you allow a per-station
   term, and does the leftover phi shrink enough to be worth the parameters?
3. **Predict which hold-outs will reverse, before running them.** If the reversal is about a test
   set sitting outside the region the model was trained in, then some measure of how far a
   country's records sit from the rest of the file — in magnitude, distance and Vs30 together —
   should order the country hold-outs by their gap. Build that measure, commit to the ordering,
   and only then run the hold-outs. A rule that predicts in advance is worth ten that explain
   afterwards.
4. **Ask how much of sigma is even reducible.** Two recordings of the same earthquake at two
   stations 500 m apart on the same rock still differ. Find such pairs in this file — same event,
   nearly the same distance, nearly the same Vs30 — and measure how far apart they are. Whatever
   that number is, no model with these columns can do better than it.

And one that is bigger than a semester. Sigma has barely moved in forty years of ground-motion
research, across an enormous increase in data and model complexity. The fit at the top of this
notebook gives 0.445 in log10 units, a factor of 2.8 in shaking, and
*Your turn 7* splits it into a between-earthquake and a within-earthquake half. If that number is
close to irreducible with the observations that exist, then the interesting question is not which
model predicts shaking best, but what would have to be *measured* — and not modelled — to make it
smaller. Answering that with a number, even a rough one, would be a real result.

### ✏️ Your turn 8 — the first move

Before you close this notebook: in a few sentences, name the **one** measurement you would make
first, say what it would show if a flexible model genuinely improves ground-motion prediction, what
it would show if it does not, and name the number that would change your mind. Then make it, in the
cell below the prose.

*(Double-click this cell and replace this line with your answer.)*

In [ ]:
# ← your answer here

